# 双膝骨刺（OST）占比

每张图在全图上统计标签 **骨刺**；**可视化仅叠色与框选骨刺类**，髌骨主体（如 `Patella`）不参与半透明叠图与 bbox。

命名示例：`KOA01_L.nrrd` 与 `KOA01_R.nrrd`（后缀可在 `osteophyte_config` 中改）。


## 环境与导入

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

from IPython.display import display

_nb = Path.cwd().resolve()
KNEE_PKG_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
if str(KNEE_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(KNEE_PKG_ROOT))

from koa.configs.osteophyte_config import (
    CURRENT_OSTEOPHYTE_CONFIG_KEY,
    OSTEOPHYTE_CONFIGS,
)
from koa.utils.sitk_utils import load_sitk_image
from koa.osteophyte import osteophyte_ratios_lr_files_auto
from koa.utils.plot_text import sanitize_plot_text
from koa.utils.bilateral_viz import plot_lr_knee_images_overlay
from koa.utils.case_list import (
    find_mask_path,
    find_volume_path,
    list_osteophyte_lr_pairs_from_config,
)


def sitk_to_2d(arr_img):
    """SimpleITK 读入的体数据取第 0 层为 2D（本笔记内单例与批量共用）。"""
    arr = sitk.GetArrayFromImage(arr_img)
    if arr.ndim == 3:
        arr = arr[0]
    return np.asarray(arr)


## 数据路径与标签

编辑 `koa/configs/osteophyte_config.py` 中的 `image_dir`、`mask_dir`、`*_suffix`、`label_mapping`（`Patella` / `Patella_Osteophyte`）等；本单元用 `dict(OSTEOPHYTE_CONFIGS[CURRENT_OSTEOPHYTE_CONFIG_KEY])` 读入当前条目。


In [ ]:
ost_cfg = dict(OSTEOPHYTE_CONFIGS[CURRENT_OSTEOPHYTE_CONFIG_KEY])
DEMO_KEY = "case"  # 可填 case_id / pair_id / 某一侧 stem
EXTS = (str(ost_cfg.get("file_type", ".nrrd")),)
TIE_OST_HI = ost_cfg.get("tie_osteophyte_is_higher_id", True)

# Demo / batch outputs: default paths from ost_cfg (same as scripts/osteophyte.py batch).
SAVE_CSV = True
SAVE_PLOTS_DEMO = True
SAVE_PLOTS_BATCH = True
_csv_default = Path(ost_cfg["output_csv"]).expanduser().resolve()
CSV_OUT = _csv_default
_raw_ost_fig = ost_cfg.get("output_figure_dir")
FIG_DIR = (
    Path(_raw_ost_fig).expanduser().resolve()
    if _raw_ost_fig
    else _csv_default.parent
)
# Override CSV_OUT / FIG_DIR below if needed.
FIG_EXT = "png"
FIG_DPI = 150

mask_dir = Path(ost_cfg["mask_dir"])
image_dir = Path(ost_cfg["image_dir"])

pairs = list_osteophyte_lr_pairs_from_config(ost_cfg)
if not pairs:
    raise FileNotFoundError(f"未在 {mask_dir} 找到 L/R 成对的 mask")

pair = pairs[0]
for p in pairs:
    if DEMO_KEY in (p["case_id"], p["pair_id"], p["left_stem"], p["right_stem"]):
        pair = p
        break

base_id = pair["case_id"]
left_stem = pair["left_stem"]
right_stem = pair["right_stem"]
_ml = find_mask_path(mask_dir, left_stem, EXTS)
_mr = find_mask_path(mask_dir, right_stem, EXTS)
_il = find_volume_path(image_dir, left_stem, EXTS)
_ir = find_volume_path(image_dir, right_stem, EXTS)
print("demo pair:", pair["pair_id"])
print("left/right stems:", left_stem, right_stem)
print("mask L/R:", _ml, _mr)
print("image L/R:", _il, _ir)


## 单例：加载、计算、双图并排显示

下一节为可选的演示图保存；批量 CSV 与批量出图各占单独代码格。


In [ ]:
if _ml is None or _mr is None or _il is None or _ir is None:
    raise FileNotFoundError("请修改 DEMO_KEY 或 ost_cfg 中的目录，使 L/R 影像与 mask 均存在")

image_l = sitk_to_2d(load_sitk_image(_il))
image_r = sitk_to_2d(load_sitk_image(_ir))
mask_l = sitk_to_2d(load_sitk_image(_ml)).astype(np.int32)
mask_r = sitk_to_2d(load_sitk_image(_mr)).astype(np.int32)

rr, rl = osteophyte_ratios_lr_files_auto(
    mask_l,
    mask_r,
    ost_cfg,
    tie_osteophyte_is_higher_id=TIE_OST_HI,
)
_pair_tag = sanitize_plot_text(str(pair["pair_id"]), fallback="subject")
_suptitle_en = (
    f"OST (osteophyte) — right | left — {_pair_tag}"
    if _pair_tag
    else "OST (osteophyte) — right | left"
)

fig = plot_lr_knee_images_overlay(
    image_l,
    mask_l,
    rl.percentage,
    image_r,
    mask_r,
    rr.percentage,
    [rr.osteophyte_label, rl.osteophyte_label],
    overlay_label_values_right=[rr.osteophyte_label],
    overlay_label_values_left=[rl.osteophyte_label],
    suptitle=_suptitle_en,
    subtitle_left="Left knee",
    subtitle_right="Right knee",
    bbox_label_ids_right=[rr.osteophyte_label],
    bbox_label_ids_left=[rl.osteophyte_label],
    legend_overlay_label="Osteophyte overlay",
)

# Debug: axes positions to quantify left/right gap
_axes = sorted(fig.get_axes(), key=lambda ax: ax.get_position().x0)
if len(_axes) >= 2:
    _p0 = _axes[0].get_position()
    _p1 = _axes[1].get_position()
    _gap_x = _p1.x0 - _p0.x1
    print("[OST demo] axes[0] bounds:", _p0.bounds)
    print("[OST demo] axes[1] bounds:", _p1.bounds)
    print("[OST demo] gap_x (fig coords):", _gap_x)
else:
    print("[OST demo] unexpected axes count:", len(_axes))

plt.show()

print("左膝：骨刺标签", rl.osteophyte_label, "|OST|", rl.osteophyte_pixels, "|髌骨|", rl.patella_pixels, "%", rl.percentage)
print("右膝：骨刺标签", rr.osteophyte_label, "|OST|", rr.osteophyte_pixels, "|髌骨|", rr.patella_pixels, "%", rr.percentage)


## 保存单例演示图（可选）

上一格已生成 `fig`。当 `SAVE_PLOTS_DEMO` 为 True 时运行本格，将图写入 `FIG_DIR`（默认来自配置 `output_figure_dir`，否则为 `output_csv` 的父目录；可在「数据路径」格覆盖 `FIG_DIR`）。

In [ ]:
if SAVE_PLOTS_DEMO:
    _csv_base = Path(CSV_OUT or ost_cfg["output_csv"]).resolve()
    _ost_fig_dir = Path(FIG_DIR).resolve() if FIG_DIR else _csv_base.parent
    _ost_fig_dir.mkdir(parents=True, exist_ok=True)
    _fig_out = _ost_fig_dir / f"{pair['pair_id']}.{str(FIG_EXT).lstrip('.') or 'png'}"
    fig.savefig(_fig_out, dpi=FIG_DPI, bbox_inches="tight")
    print("Saved figure:", _fig_out.resolve())
else:
    print("SAVE_PLOTS_DEMO=False，跳过写图")

## 批量：写 CSV 与批量出图

- **下一格**：遍历配对，汇总 `df_ost`，按 `SAVE_CSV` 写 CSV。
- **再下一格**：`SAVE_PLOTS_BATCH=True` 时为每对保存双膝叠图（需能解析到 L/R 影像路径）。

配对规则：`<case_id>_<L/R>_<AGE>_<M/F>[_0000]` 在 `mask_dir` 中配对（不依赖 `meta_data_csv`）。命令行：`python scripts/osteophyte.py --csv-only`。


In [ ]:
pairs_all = list_osteophyte_lr_pairs_from_config(ost_cfg)
print("batch CSV: total pairs from config =", len(pairs_all))
missing_left = 0
missing_right = 0
missing_either = 0

rows = []
_csv_base = Path(CSV_OUT or ost_cfg["output_csv"]).resolve()

for pair_b in pairs_all:
    left_stem = pair_b["left_stem"]
    right_stem = pair_b["right_stem"]
    pl = find_mask_path(mask_dir, left_stem, EXTS)
    pr = find_mask_path(mask_dir, right_stem, EXTS)
    if pl is None or pr is None:
        if pl is None:
            missing_left += 1
        if pr is None:
            missing_right += 1
        missing_either += 1
        continue

    m_l = sitk_to_2d(load_sitk_image(pl)).astype(np.int32)
    m_r = sitk_to_2d(load_sitk_image(pr)).astype(np.int32)
    r_r, r_l = osteophyte_ratios_lr_files_auto(
        m_l, m_r, ost_cfg, tie_osteophyte_is_higher_id=TIE_OST_HI
    )
    rows.append(
        {
            "case_id": pair_b["case_id"],
            "right_osteophyte_label": r_r.osteophyte_label,
            "right_patella_body_label": r_r.patella_label_other,
            "right_osteophyte_pixels": r_r.osteophyte_pixels,
            "right_patella_pixels": r_r.patella_pixels,
            "right_osteophyte_pct_of_patella": r_r.percentage,
            "left_osteophyte_label": r_l.osteophyte_label,
            "left_patella_body_label": r_l.patella_label_other,
            "left_osteophyte_pixels": r_l.osteophyte_pixels,
            "left_patella_pixels": r_l.patella_pixels,
            "left_osteophyte_pct_of_patella": r_l.percentage,
        }
    )

df_ost = pd.DataFrame(rows)
print(
    f"batch CSV: kept rows={len(df_ost)}; missing_either={missing_either} "
    f"(missing_left={missing_left}, missing_right={missing_right})"
)

if SAVE_CSV:
    out_csv = _csv_base
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_ost.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"Saved {len(df_ost)} rows -> {out_csv}")
else:
    print(f"Computed {len(df_ost)} pairs (SAVE_CSV=False; CSV not written)")

display(df_ost.head())


## 批量保存叠图（可选）

需已运行 **环境与导入**、**数据路径与标签** 两格（与 `sitk_to_2d` 定义）。不必先跑 CSV 格。`SAVE_PLOTS_BATCH=True` 时为每对 L/R 保存一张双膝图；缺影像路径的 pair 会跳过。

In [ ]:
_csv_base = Path(CSV_OUT or ost_cfg["output_csv"]).resolve()
_fig_dir_batch = Path(FIG_DIR).resolve() if FIG_DIR else _csv_base.parent
_fig_dir_batch.mkdir(parents=True, exist_ok=True)
_ext_batch = str(FIG_EXT).lstrip(".") or "png"

if not SAVE_PLOTS_BATCH:
    print("SAVE_PLOTS_BATCH=False，跳过批量出图")
else:
    n_saved = 0
    for pair_b in list_osteophyte_lr_pairs_from_config(ost_cfg):
        left_stem = pair_b["left_stem"]
        right_stem = pair_b["right_stem"]
        pl = find_mask_path(mask_dir, left_stem, EXTS)
        pr = find_mask_path(mask_dir, right_stem, EXTS)
        if pl is None or pr is None:
            continue
        m_l = sitk_to_2d(load_sitk_image(pl)).astype(np.int32)
        m_r = sitk_to_2d(load_sitk_image(pr)).astype(np.int32)
        r_r, r_l = osteophyte_ratios_lr_files_auto(
            m_l, m_r, ost_cfg, tie_osteophyte_is_higher_id=TIE_OST_HI
        )
        il = find_volume_path(image_dir, left_stem, EXTS)
        ir = find_volume_path(image_dir, right_stem, EXTS)
        if il is None or ir is None:
            continue
        image_l_b = sitk_to_2d(load_sitk_image(il))
        image_r_b = sitk_to_2d(load_sitk_image(ir))
        _tag_b = sanitize_plot_text(str(pair_b["pair_id"]), fallback="subject")
        _sup_b = (
            f"OST (osteophyte) — right | left — {_tag_b}"
            if _tag_b
            else "OST (osteophyte) — right | left"
        )
        fig_b = plot_lr_knee_images_overlay(
            image_l_b,
            m_l,
            r_l.percentage,
            image_r_b,
            m_r,
            r_r.percentage,
            [r_r.osteophyte_label, r_l.osteophyte_label],
            overlay_label_values_right=[r_r.osteophyte_label],
            overlay_label_values_left=[r_l.osteophyte_label],
            suptitle=_sup_b,
            subtitle_left="Left knee",
            subtitle_right="Right knee",
            bbox_label_ids_right=[r_r.osteophyte_label],
            bbox_label_ids_left=[r_l.osteophyte_label],
            legend_overlay_label="Osteophyte overlay",
        )

        if n_saved == 0:
            _axes = sorted(fig_b.get_axes(), key=lambda ax: ax.get_position().x0)
            if len(_axes) >= 2:
                _p0 = _axes[0].get_position()
                _p1 = _axes[1].get_position()
                _gap_x = _p1.x0 - _p0.x1
                print("[OST batch] axes[0] bounds:", _p0.bounds)
                print("[OST batch] axes[1] bounds:", _p1.bounds)
                print("[OST batch] gap_x (fig coords):", _gap_x)
            else:
                print("[OST batch] unexpected axes count:", len(_axes))

        _out_b = _fig_dir_batch / f"{pair_b['pair_id']}.{_ext_batch}"
        fig_b.savefig(_out_b, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig_b)
        n_saved += 1
    print(f"Batch figures -> {_fig_dir_batch} (*.{_ext_batch}), saved {n_saved} files")